# FDA Approved Drugs — Bulk Export to CSV

Downloads the full openFDA **Drugs@FDA** bulk dataset and writes out a CSV of all FDA-approved drug names (brand + generic), deduplicated.

- Source: https://open.fda.gov/apis/drug/drugsfda/
- Bulk download index: https://api.fda.gov/download.json

**Notes:**
- Requires outbound internet access to `download.open.fda.gov`.
- Covers products approved under NDAs/ANDAs/BLAs submitted to FDA's Center for Drug Evaluation and Research (small molecules; biologics licensed under CBER are **not** included here — see the Purple Book for those).
- Each *application* can have multiple *products* (different strengths/dosage forms), and multiple applications can share the same brand or generic name (e.g. many generic manufacturers of "furosemide"). This notebook dedupes at the `(brand_name, generic_name)` level.
- Includes discontinued/withdrawn products by default — set `ACTIVE_ONLY = True` in the config cell to keep only currently marketed products.

In [1]:
import csv
import io
import json
import zipfile
from urllib.request import urlopen, Request

BULK_INDEX_URL = "https://api.fda.gov/download.json"
ACTIVE_ONLY = False  # set True to drop marketing_status == "Discontinued" / "None (Tentative Approval)"
OUTPUT_CSV = "fda_approved_drugs.csv"

## 1. Look up the current bulk file URL(s)

In [2]:
def get_bulk_file_urls():
    """Look up current Drugs@FDA bulk JSON zip file URL(s) from FDA's download index.
    Falls back to the known single-file URL if the index format changes."""
    try:
        req = Request(BULK_INDEX_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(req, timeout=60) as resp:
            index = json.load(resp)
        return [
            p["file"]
            for p in index["results"]["drug"]["drugsfda"]["partitions"]
        ]
    except Exception as e:
        print(f"Falling back to known static URL (index lookup failed: {e})")
        return ["https://download.open.fda.gov/drug/drugsfda/drug-drugsfda-0001-of-0001.json.zip"]

urls = get_bulk_file_urls()
print(f"Found {len(urls)} bulk file(s) to download.")
urls

Found 1 bulk file(s) to download.


['https://download.open.fda.gov/drug/drugsfda/drug-drugsfda-0001-of-0001.json.zip']

## 2. Download and stream-parse each bulk file

In [3]:
def iter_records(zip_url):
    req = Request(zip_url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(req, timeout=300) as resp:
        data = resp.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        json_name = [n for n in zf.namelist() if n.endswith(".json")][0]
        with zf.open(json_name) as f:
            payload = json.load(f)
    yield from payload["results"]

## 3. Collect and dedupe drug names

In [4]:
rows = {}  # (brand_name, generic_name) -> row dict

for url in urls:
    print(f"Downloading {url} ...")
    count_before = len(rows)
    for rec in iter_records(url):
        app_no = rec.get("application_number", "")
        sponsor = rec.get("sponsor_name", "")
        openfda = rec.get("openfda", {}) or {}
        brand_names = openfda.get("brand_name") or [None]
        generic_names = openfda.get("generic_name") or [None]

        for product in rec.get("products", []):
            brand = product.get("brand_name") or (brand_names[0] if brand_names else None)
            status = product.get("marketing_status", "")
            if ACTIVE_ONLY and status and "discontinued" in status.lower():
                continue
            ingredients = product.get("active_ingredients", [])
            generic = "; ".join(sorted({ai.get("name", "") for ai in ingredients if ai.get("name")}))
            if not generic and generic_names:
                generic = generic_names[0] or ""

            key = (brand or "", generic or "")
            if key == ("", ""):
                continue
            if key not in rows:
                rows[key] = {
                    "brand_name": brand or "",
                    "generic_name": generic or "",
                    "application_number": app_no,
                    "sponsor_name": sponsor,
                    "dosage_form": product.get("dosage_form", ""),
                    "route": product.get("route", ""),
                    "marketing_status": status,
                }
    print(f"  -> {len(rows) - count_before} new unique entries from this file")

print(f"\nCollected {len(rows)} unique drug name entries total.")

  -> 8534 new unique entries from this file

Collected 8534 unique drug name entries total.


## 4. Preview as a DataFrame (optional)

In [5]:
import pandas as pd

df = pd.DataFrame(sorted(rows.values(), key=lambda r: (r["brand_name"], r["generic_name"])))
df.head(20)

,brand_name,generic_name,application_number,sponsor_name,dosage_form,route,marketing_status
0,8-HOUR BAYER,ASPIRIN,NDA016030,BAYER,"TABLET, EXTENDED RELEASE",ORAL,Discontinued
1,8-MOP,METHOXSALEN,NDA009048,VALEANT PHARM INTL,CAPSULE,ORAL,Discontinued
2,A-HYDROCORT,HYDROCORTISONE SODIUM SUCCINATE,ANDA089577,ABBOTT,INJECTABLE,INJECTION,Discontinued
3,A-METHAPRED,METHYLPREDNISOLONE SODIUM SUCCINATE,ANDA085852,HOSPIRA,INJECTABLE,INJECTION,Discontinued
4,A-POXIDE,CHLORDIAZEPOXIDE HYDROCHLORIDE,ANDA085513,ABBOTT,CAPSULE,ORAL,Discontinued
5,A.P.L.,"GONADOTROPIN, CHORIONIC",BLA017055,FERRING,INJECTABLE,INJECTION,Discontinued
6,A/T/S,ERYTHROMYCIN,ANDA062405,TARO,SOLUTION,TOPICAL,Discontinued
7,ABACAVIR AND LAMIVUDINE,ABACAVIR; LAMIVUDINE,NDA208775,CIPLA LIMITED,"TABLET, FOR SUSPENSION",ORAL,None (Tentative Approval)
8,ABACAVIR SULFATE,ABACAVIR SULFATE,ANDA201107,HETERO LABS LTD III,SOLUTION,ORAL,Prescription
9,ABACAVIR SULFATE AND LAMIVUDINE,ABACAVIR SULFATE; LAMIVUDINE,ANDA212663,MACLEODS PHARMS,TABLET,ORAL,Discontinued


## 5. Write out the CSV

In [6]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(df)} rows to {OUTPUT_CSV}")

Wrote 8534 rows to fda_approved_drugs.csv
